esse código baixa todas as bibliotecas que serão utilizadas no experimento

In [1]:
!pip install -r requirements.txt

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


# Passo 1 - Seleção dos Falsos Negativos e Verdadeiros Negativos

In [2]:
from unidecode import unidecode
import re
import unicodedata
import pandas as pd
from joblib import load
import string
from itertools import product
from sklearn.feature_extraction.text import CountVectorizer

In [3]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\kimli\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [4]:
rf_load = load(r'C:\Users\kimli\Documents\GitHub\PIBIC\corretor\rf_att.joblib')

In [5]:
def normalize_text(text):
    if pd.isna(text):
        return text
    text = unicodedata.normalize('NFKD', text)
    text = ''.join(c for c in text if not unicodedata.combining(c))
    return text.upper()

In [6]:
df_origin = pd.read_excel(r'C:\Users\kimli\Documents\GitHub\PIBIC\corretor\Bases\BOS.xlsx', usecols=['n_bop','relato']).drop_duplicates()
df_sintetic = pd.read_excel(r'C:\Users\kimli\Documents\GitHub\PIBIC\corretor\Bases\boletins_de_ocorrencia_pará (11).xlsx', usecols=['RELATO']).drop_duplicates()

In [7]:
base_relato = pd.DataFrame({'n_bop': df_origin['n_bop'] ,'relato' : pd.concat([df_origin['relato'], df_sintetic['RELATO']], ignore_index=True)})

base_relato['relato'] = base_relato['relato'].apply(normalize_text)
base_relato

,n_bop,relato
0,00615/2021.100191-1,"A DECLARANTE ACIMA QUALIFICADA, INFORMA QUE E..."
1,00616/2021.100053-9,A RELATORA ACIMA QUALIFICADA COMPARECE NESTA ...
2,00616/2021.100190-3,A RELATORA ACIMA QUALIFICADA COMPARECE NESTA ...
3,00616/2021.100222-0,A RELATORA ACIMA QUALIFICADA COMPARECE NESTA ...
4,00616/2021.100016-0,O RELATOR ACIMA QUALIFICADO COMPARECE NESTA D...
...,...,...
595,NaN,A VITIMA LAVINIA CAVALCANTI DE ANDRADE DA ROSA...
596,NaN,FOI REGISTRADO UM BOLETIM DE OCORRENCIA POR RA...
597,NaN,"A DECLARANTE, THEO SAMPAIO NOVAIS E VIEIRA GAR..."
598,NaN,A VITIMA RYAN ALBUQUERQUE E CAMARA CASA GRANDE...


In [8]:
gab = pd.read_csv(r"C:\Users\kimli\Documents\GitHub\PIBIC\corretor\Bases\joao\gabarito_atualizado.csv")

### Pré-processamento

realizando o pré-processamento removendo: Stopwords, caracteres especias, números e pontuação dos relatos.

In [9]:
def preprocess(relato):
    def remove_stop_words(relato):
        # Remove stop words em português
        sw = stopwords.words('portuguese')
        sw = [unidecode(word).upper() for word in sw]
        return ' '.join([token for token in re.split("[, ]", relato) if len(token) > 2 and token not in sw])
    
    def clean_relato(relato):
        # Remoção de números e pontuação
        relato = re.sub(r'(<[^<>]*>)|(&NBSP;)|(&nbsp;)', " ", relato)
        relato = re.sub(r"[^\w\s]", " ", relato)
        relato = re.sub(r"[\d]", " ", relato)
        relato = remove_stop_words(relato)
        return relato
    
    relato = clean_relato(relato)
    return relato

In [10]:
base_relato['relato_tratado'] = base_relato['relato'].apply(preprocess)

### Identificação dos falsos negativos

Neste trecho é realizada a identificação e criação de um DataFrame com os falsos negativos (tokens classificados como palavra '0', pelo modelo, que na verdade são nomes '1', no gabarito) após a predição dos tokens do relato relato

In [11]:
def false_negative(df_bop : pd.DataFrame):
    fn_list = []

    for n_bop, text in zip(df_bop['n_bop'], df_bop['relato_tratado']):
        text = text.split()
        df = pd.DataFrame(text, columns=['palavra'])
        
        # aqui eu realizo a predição    
        df['y_palavra'] = rf_load.predict(df['palavra'].to_list())
        
        df['predecessor'] = df['palavra'].shift(1).fillna('')
        df['y_predecessor'] = df['predecessor'].apply(lambda x: df[df['palavra'] == x]['y_palavra'].iloc[0] if x else '')
        
        df['sucessor'] = df['palavra'].shift(-1).fillna('')
        df['y_sucessor'] = df['sucessor'].apply(lambda x: df[df['palavra'] == x]['y_palavra'].iloc[0] if x else '')
        
        for _, row in df.iterrows():
            y_true_row = gab[gab['palavra'] == row['palavra']]
            y_true = y_true_row['y'].iloc[0] if not y_true_row.empty else 0
            
            if row['y_palavra'] == 0 and y_true == 1:
                fn_list.append({
                    'predecessor': row['predecessor'],
                    'y_predecessor': str(row['y_predecessor']),
                    'palavra': row['palavra'],
                    'y_palavra': str(row['y_palavra']),
                    'sucessor': row['sucessor'],
                    'y_sucessor': str(row['y_sucessor']),
                    'n_bop': n_bop
                })
    
    return pd.DataFrame(fn_list)


### Identificação dos verdadeiros negativos

Neste trecho é realizada a identificação e criação de um DataFrame com os verdadeiros negativos (tokens classificados como palavra '0', pelo modelo, e que realmente são palavras '0', no gabarito) após a predição dos tokens do relato

In [12]:
def true_negative(df_bop : pd.DataFrame):
    fn_list = []

    for n_bop, text in zip(df_bop['n_bop'], df_bop['relato_tratado']):
        text = text.split()
        df = pd.DataFrame(text, columns=['palavra'])
        
        df['y_palavra'] = rf_load.predict(df['palavra'].to_list())
        
        df['predecessor'] = df['palavra'].shift(1).fillna('')
        df['y_predecessor'] = df['predecessor'].apply(lambda x: df[df['palavra'] == x]['y_palavra'].iloc[0] if x else '')
        
        df['sucessor'] = df['palavra'].shift(-1).fillna('')
        df['y_sucessor'] = df['sucessor'].apply(lambda x: df[df['palavra'] == x]['y_palavra'].iloc[0] if x else '')
        
        for _, row in df.iterrows():
            y_true_row = gab[gab['palavra'] == row['palavra']]
            y_true = y_true_row['y'].iloc[0] if not y_true_row.empty else 0
            
            if row['y_palavra'] == 0 and y_true == 0:
                fn_list.append({
                    'predecessor': row['predecessor'],
                    'y_predecessor': str(row['y_predecessor']),
                    'palavra': row['palavra'],
                    'y_palavra': str(row['y_palavra']),
                    'sucessor': row['sucessor'],
                    'y_sucessor': str(row['y_sucessor']),
                    'n_bop': n_bop
                })
    
    return pd.DataFrame(fn_list)


In [13]:
fn_base_relatos = false_negative(base_relato)
tn_base_relatos = true_negative(base_relato)

# IMPORTANTE

Surgiu um desafio na etapa da realização da captura dos predecessores e sucessores. Foram identificados, em alguns casos, a inexistencia de um predecessor ou sucessor do token classificado. Por exemplo, 'RELATOR', que se localiza no início dos textos, é um VERDADEIRO NEGATIVO, e por ser a primeira palavra do vetor de tokens não possui predecessor. Também ocorreram problemas na captura dos FALSOS NEGATIVOS, ouve a ocorrência de dois nomes que se localizavam no fim do vetor de tokens, estas não possuiam sucessores.

Qual o problema? Devido a falta do valor do preditor acerca da palavra inexistente, ele armazena apenas '' (vazio). Isso é um problema já que os modelos utilizados trabalham apenas com dados numéricos.

Estas ocorrências, por mais raras que possa parecer, são desafios que nosso modelo deve estar preparado, visto que há sim grandes chances de ele se deparar com situações como esta em um ambiente real.

Diante disso, foi elaborada uma estratégia para tratar este erro.

Foi implementada uma nova categoria, justamente para solucionar este problema. Assim, palavras que não possuem predecessor ou sucessores, receberão como valor para os respectivos vizinhos o número inteiro '2'.

Logo, temos estas categorias para os tokens de predecessor ou sucessores:

0 - palavra  
1 - nome  
2 - predecessor ou sucessor inexistente  

In [14]:
fn_base_relatos['y_sucessor'] = fn_base_relatos['y_sucessor'].replace({'':2})
fn_base_relatos['y_predecessor'] = fn_base_relatos['y_predecessor'].replace({'':2})

In [15]:
fn_base_relatos['y_sucessor'] = fn_base_relatos['y_sucessor'].astype(int)
fn_base_relatos['y_predecessor'] = fn_base_relatos['y_predecessor'].astype(int)
fn_base_relatos['y_palavra'] = fn_base_relatos['y_palavra'].astype(int)

In [16]:
fn_base_relatos

,predecessor,y_predecessor,palavra,y_palavra,sucessor,y_sucessor,n_bop
0,PERFIL,0,PEREIRARENATA,0,SABE,1,00616/2021.100190-3
1,WILLIAN,1,VASMORBITA,0,BANCO,0,00615/2021.100534-8
2,CODIGO,0,BARRA,0,BANCO,0,00615/2021.100729-8
3,MARCOS,1,BARRETO,0,CNPJ,1,00615/2021.100729-8
4,PIX,0,CHAVE,0,CPF,1,00615/2021.100813-1
...,...,...,...,...,...,...,...
419,ISADORA,1,CAMPOS,0,MONTENEGRO,0,NaN
420,VIADUTO,1,ANA,0,CECILIA,1,NaN
421,CAMPO,0,FONSECA,0,APTO,0,NaN
422,LAVINIA,1,CAVALCANTI,0,ANDRADE,1,NaN


Está é a etapa a qual é realizada o tratamento dos predecessores e sucessores inexistentes

In [17]:
tn_base_relatos['y_sucessor'] = tn_base_relatos['y_sucessor'].replace({'':2})
tn_base_relatos['y_predecessor'] = tn_base_relatos['y_predecessor'].replace({'':2})

In [18]:
tn_base_relatos['y_sucessor'] = tn_base_relatos['y_sucessor'].astype(int)
tn_base_relatos['y_predecessor'] = tn_base_relatos['y_predecessor'].astype(int)
tn_base_relatos['y_palavra'] = tn_base_relatos['y_palavra'].astype(int)

In [19]:
tn_base_relatos

,predecessor,y_predecessor,palavra,y_palavra,sucessor,y_sucessor,n_bop
0,,2,DECLARANTE,0,ACIMA,0,00615/2021.100191-1
1,DECLARANTE,0,ACIMA,0,QUALIFICADA,0,00615/2021.100191-1
2,ACIMA,0,QUALIFICADA,0,INFORMA,0,00615/2021.100191-1
3,QUALIFICADA,0,INFORMA,0,PROFESSORA,0,00615/2021.100191-1
4,INFORMA,0,PROFESSORA,0,LINGUA,1,00615/2021.100191-1
...,...,...,...,...,...,...,...
36029,ORG,1,TELEFONE,0,BANCO,0,NaN
36030,TELEFONE,0,BANCO,0,AFETADO,0,NaN
36031,BANCO,0,AFETADO,0,BANCO,0,NaN
36032,AFETADO,0,BANCO,0,PAN,1,NaN


### Seleção dos VERDADEIROS NEGATIVOS para o balanceamento da base

Aqui é realizada a seleção pseudo-aleatória das amostras da base de VERDADEIROS NEGATIVOS, isso é realizado pelo fato da base de verdadeiros negativos ser muito superior a base de FALSOS NEGATIVOS.  

Logo, para que ambas as bases possuam o mesmo número de amostras, foi realizada a extração 'aleatória' de 406  amostras da base de VERDADEIROS NEGATIVOS.

In [20]:
tn_base_relatos = tn_base_relatos.sample(n=len(fn_base_relatos), random_state=42)
tn_base_relatos = tn_base_relatos.reset_index(drop=True)
tn_base_relatos

,predecessor,y_predecessor,palavra,y_palavra,sucessor,y_sucessor,n_bop
0,NAMORADA,0,DECLARANTE,0,TENDO,0,00615/2021.100094-9
1,AZUL,0,VARIAS,0,JOIAS,1,00004/2021.110797-7
2,CASOS,0,PERFIL,0,ENTROU,0,NaN
3,VITIMA,0,GOLPE,0,FALSO,0,NaN
4,UTILIZACAO,0,CONDICAO,0,PESSOA,0,613/2021.000009-8
...,...,...,...,...,...,...,...
419,SIM,0,ALGUEM,0,DESCONHECIDO,0,00615/2021.101102-4
420,BANCARIOS,0,ROUBADOS,0,APOS,0,NaN
421,HORAS,0,RELATA,0,VERIFICOU,0,00615/2021.100857-1
422,ENDERECO,0,RECEBER,0,PRODUTO,0,00031/2021.102292-0


# Passo 2 - Criação do Vetor de Bigramas dos Falsos Negativos e Verdadeiros Negativos

Este é o mesmo modelo de vetor de bigramas dos experimentos anteriores, podes visualizar logo mais a frente.

In [21]:
def bigrams_vector(lista_fn):
    alfabeto = list(string.ascii_uppercase)
    alfabeto_bigrams = [''.join(bigram) for bigram in list(product(alfabeto, repeat=2))]
    vectorizer = CountVectorizer(analyzer='char', vocabulary=alfabeto_bigrams, lowercase=False, ngram_range=(2, 2))
    vector = vectorizer.fit_transform(lista_fn)
    return pd.DataFrame(vector.todense(), columns=alfabeto_bigrams)

In [22]:
palavra = fn_base_relatos['palavra']
fn_vector_bigrams = bigrams_vector(palavra.to_list())
fn_vector_bigrams

,AA,AB,AC,AD,AE,AF,AG,AH,AI,AJ,...,ZQ,ZR,ZS,ZT,ZU,ZV,ZW,ZX,ZY,ZZ
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
419,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
420,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
421,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
422,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [23]:
palavra = tn_base_relatos['palavra']
tn_vector_bigrams = bigrams_vector(palavra.to_list())
tn_vector_bigrams

,AA,AB,AC,AD,AE,AF,AG,AH,AI,AJ,...,ZQ,ZR,ZS,ZT,ZU,ZV,ZW,ZX,ZY,ZZ
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
419,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
420,0,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
421,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
422,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [24]:
fn_vector_bigrams = pd.concat([fn_vector_bigrams, fn_base_relatos[['predecessor', 'y_predecessor', 'palavra','sucessor','y_sucessor']]], axis=1)
fn_vector_bigrams

,AA,AB,AC,AD,AE,AF,AG,AH,AI,AJ,...,ZV,ZW,ZX,ZY,ZZ,predecessor,y_predecessor,palavra,sucessor,y_sucessor
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,PERFIL,0,PEREIRARENATA,SABE,1
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,WILLIAN,1,VASMORBITA,BANCO,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,CODIGO,0,BARRA,BANCO,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,MARCOS,1,BARRETO,CNPJ,1
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,PIX,0,CHAVE,CPF,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
419,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,ISADORA,1,CAMPOS,MONTENEGRO,0
420,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,VIADUTO,1,ANA,CECILIA,1
421,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,CAMPO,0,FONSECA,APTO,0
422,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,LAVINIA,1,CAVALCANTI,ANDRADE,1


In [25]:
tn_vector_bigrams = pd.concat([tn_vector_bigrams, tn_base_relatos[['predecessor', 'y_predecessor', 'palavra','sucessor','y_sucessor']]], axis=1)
tn_vector_bigrams

,AA,AB,AC,AD,AE,AF,AG,AH,AI,AJ,...,ZV,ZW,ZX,ZY,ZZ,predecessor,y_predecessor,palavra,sucessor,y_sucessor
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,NAMORADA,0,DECLARANTE,TENDO,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,AZUL,0,VARIAS,JOIAS,1
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,CASOS,0,PERFIL,ENTROU,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,VITIMA,0,GOLPE,FALSO,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,UTILIZACAO,0,CONDICAO,PESSOA,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
419,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,SIM,0,ALGUEM,DESCONHECIDO,0
420,0,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,BANCARIOS,0,ROUBADOS,APOS,0
421,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,HORAS,0,RELATA,VERIFICOU,0
422,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,ENDERECO,0,RECEBER,PRODUTO,0


# Passo 3 - Realização da Classificação Gramatical dos Falsos Negativos e Verdadeiros Positivos

In [26]:
import spacy as scy
import pandas as pd
from sklearn.preprocessing import LabelEncoder

pra utilizar o módulo de classificação gramatical do spacy precisa realizar esse download logo abaixo. Demora pra dedéu :(

In [27]:
nlp = scy.load('pt_core_news_lg')

In [28]:
lb = LabelEncoder()

# Criação dos labels da classificação gramatical

No código que aproveitei do classificador do spacy, possuia a renomeação dos labels para 'pt-br', digamos assim. Por essa questão, não alterei já que a visualização se torna bem melhor para validação dos resultados exibidos pelo classificador, então foram criadas 14 labels para a classificação, estes estão listado abaixo:

In [29]:
labels = ['<SUBSTANTIVO>', '<VERBO>', '<ADJETIVO>', '<ADVERBIO>', '<PONTUACAO>',
            '<CONJUNCAO>', '<PRONOME>', '<ARTIGO>', '<PREPOSIÇÃO>', '<NUMERAL>',
            '<SIMBOLO>', '<PALAVRA ESTRANGEIRA>', '<ESPACO>', '<NULL>']

In [30]:
lb.fit(labels)

LabelEncoder()

In [31]:
lb.classes_

array(['<ADJETIVO>', '<ADVERBIO>', '<ARTIGO>', '<CONJUNCAO>', '<ESPACO>',
       '<NULL>', '<NUMERAL>', '<PALAVRA ESTRANGEIRA>', '<PONTUACAO>',
       '<PREPOSIÇÃO>', '<PRONOME>', '<SIMBOLO>', '<SUBSTANTIVO>',
       '<VERBO>'], dtype='<U21')

esta é a função que realiza a renomeação

In [32]:
def renomear(doc):

    csf = []
    for token in doc:
        if token.pos_ == 'NOUN' or token.pos_ == 'PROPN':
            csf.append('<SUBSTANTIVO>')
        elif token.pos_ == 'VERB' or token.pos_ == 'AUX':
            csf.append('<VERBO>')
        elif token.pos_ == 'ADJ':
            csf.append('<ADJETIVO>')
        elif token.pos_ == 'ADV':
            csf.append('<ADVERBIO>')
        elif token.pos_ == 'PUNCT':
            csf.append('<PONTUACAO>')
        elif token.pos_ == 'CCONJ' or token.pos_ == 'SCONJ':
            csf.append('<CONJUNCAO>')
        elif token.pos_ == 'PRON':
            csf.append('<PRONOME>')
        elif token.pos_ == 'DET':
            csf.append('<ARTIGO>')
        elif token.pos_ == 'ADP':
            csf.append('<PREPOSIÇÃO>')
        elif token.pos_ == 'NUM':
            csf.append('<NUMERAL>')
        elif token.pos_ == 'SYM':
            csf.append('<SIMBOLO>')
        elif token.pos_ == 'X':
            csf.append('<PALAVRA ESTRANGEIRA>')
        elif token.pos_ == 'SPACE':
            csf.append('<ESPACO>')
    return csf


# Tratamento dos valores vazios de predecessores e sucessores

Sim, teve dor de cabeça para os predecessores e sucessores inexistentes aqui também. Este foi solucionado de maneira mais simples.

Se observar na lista anterior de labels, podes identificar um label que não é encontrado na função de renomear, aqui em cima. Esse carinha é o <NULL>

Esse cara será a nossa classificação gramatical para os nulos, já que não tem como classificar os mesmos com nada além disso, já que eles não existem.

Logo, aqui a baixo podes ler:

Se token == '':
    adicione na lista word_class <NULL>

tratando os nossos predecessores e sucessores vazios

In [33]:
def create_base_cg(lista_tokens):
    word_class = []
    for token in lista_tokens:
        if token == '':
            word_class.append('<NULL>')
        else:
            lreview = token.lower()
            ldoc = nlp(lreview)
            classes = renomear(ldoc)
            for i in range(len(token.split())):
                word_class.append(classes[i])

    return pd.DataFrame({'palavra':lista_tokens, 'classe':word_class})

Aqui é onde realizo a chamada da função que irá realizar a classificação de todo mundo, tanto da palavra, quanto do predecessor e sucessor. Adicionando a mesma juntamente com o vetor de bigramas e a predição do predecessor e sucessor.

In [34]:
fn_vector_bigrams['cg_p'] = create_base_cg(fn_base_relatos['predecessor'])['classe']
fn_vector_bigrams['cg_w'] = create_base_cg(fn_base_relatos['palavra'])['classe']
fn_vector_bigrams['cg_s'] = create_base_cg(fn_base_relatos['sucessor'])['classe']

In [35]:
fn_vector_bigrams['cg_p'] = lb.transform(fn_vector_bigrams['cg_p'])
fn_vector_bigrams['cg_w'] = lb.transform(fn_vector_bigrams['cg_w'])
fn_vector_bigrams['cg_s'] = lb.transform(fn_vector_bigrams['cg_s'])

In [36]:
fn_vector_bigrams

,AA,AB,AC,AD,AE,AF,AG,AH,AI,AJ,...,ZY,ZZ,predecessor,y_predecessor,palavra,sucessor,y_sucessor,cg_p,cg_w,cg_s
0,0,0,0,0,0,0,0,0,0,0,...,0,0,PERFIL,0,PEREIRARENATA,SABE,1,12,0,13
1,0,0,0,0,0,0,0,0,0,0,...,0,0,WILLIAN,1,VASMORBITA,BANCO,0,12,13,12
2,0,0,0,0,0,0,0,0,0,0,...,0,0,CODIGO,0,BARRA,BANCO,0,12,12,12
3,0,0,0,0,0,0,0,0,0,0,...,0,0,MARCOS,1,BARRETO,CNPJ,1,12,12,13
4,0,0,0,0,0,0,0,0,0,0,...,0,0,PIX,0,CHAVE,CPF,1,12,12,12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
419,0,0,0,0,0,0,0,0,0,0,...,0,0,ISADORA,1,CAMPOS,MONTENEGRO,0,12,12,0
420,0,0,0,0,0,0,0,0,0,0,...,0,0,VIADUTO,1,ANA,CECILIA,1,12,12,12
421,0,0,0,0,0,0,0,0,0,0,...,0,0,CAMPO,0,FONSECA,APTO,0,12,12,0
422,0,0,0,0,0,0,0,0,0,0,...,0,0,LAVINIA,1,CAVALCANTI,ANDRADE,1,12,13,12


In [37]:
tn_vector_bigrams['cg_p'] = create_base_cg(tn_base_relatos['predecessor'])['classe']
tn_vector_bigrams['cg_w'] = create_base_cg(tn_base_relatos['palavra'])['classe']
tn_vector_bigrams['cg_s'] = create_base_cg(tn_base_relatos['sucessor'])['classe']

In [38]:
tn_vector_bigrams['cg_p'] = lb.transform(tn_vector_bigrams['cg_p'])
tn_vector_bigrams['cg_w'] = lb.transform(tn_vector_bigrams['cg_w'])
tn_vector_bigrams['cg_s'] = lb.transform(tn_vector_bigrams['cg_s'])

In [39]:
tn_vector_bigrams

,AA,AB,AC,AD,AE,AF,AG,AH,AI,AJ,...,ZY,ZZ,predecessor,y_predecessor,palavra,sucessor,y_sucessor,cg_p,cg_w,cg_s
0,0,0,0,0,0,0,0,0,0,0,...,0,0,NAMORADA,0,DECLARANTE,TENDO,0,0,0,13
1,0,0,0,0,0,0,0,0,0,0,...,0,0,AZUL,0,VARIAS,JOIAS,1,0,12,12
2,0,0,0,0,0,0,0,0,0,0,...,0,0,CASOS,0,PERFIL,ENTROU,0,12,12,13
3,0,0,0,0,0,0,0,0,0,0,...,0,0,VITIMA,0,GOLPE,FALSO,0,0,12,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,UTILIZACAO,0,CONDICAO,PESSOA,0,12,12,12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
419,0,0,0,0,0,0,0,0,0,0,...,0,0,SIM,0,ALGUEM,DESCONHECIDO,0,1,10,13
420,0,0,0,1,0,0,0,0,0,0,...,0,0,BANCARIOS,0,ROUBADOS,APOS,0,12,13,12
421,0,0,0,0,0,0,0,0,0,0,...,0,0,HORAS,0,RELATA,VERIFICOU,0,12,13,13
422,0,0,0,0,0,0,0,0,0,0,...,0,0,ENDERECO,0,RECEBER,PRODUTO,0,12,13,12


In [40]:
for index, classe in zip(range(0, len(lb.classes_)), lb.classes_):
    print(str(index) + ' ' + classe)

0 <ADJETIVO>
1 <ADVERBIO>
2 <ARTIGO>
3 <CONJUNCAO>
4 <ESPACO>
5 <NULL>
6 <NUMERAL>
7 <PALAVRA ESTRANGEIRA>
8 <PONTUACAO>
9 <PREPOSIÇÃO>
10 <PRONOME>
11 <SIMBOLO>
12 <SUBSTANTIVO>
13 <VERBO>


# drop duplicates

salvar em excel e olhar na mão se realmente é tudo isso

In [41]:
fn_vector_bigrams.shape

(424, 684)

In [42]:
fn_vector_bigrams = fn_vector_bigrams.drop_duplicates()
fn_vector_bigrams.shape

(285, 684)

In [43]:
fn_base_final = fn_vector_bigrams.drop(['sucessor', 'predecessor','palavra'], axis=1)
fn_base_final

,AA,AB,AC,AD,AE,AF,AG,AH,AI,AJ,...,ZV,ZW,ZX,ZY,ZZ,y_predecessor,y_sucessor,cg_p,cg_w,cg_s
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,12,0,13
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,12,13,12
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,12,12,12
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,1,12,12,13
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,12,12,12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
419,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,12,12,0
420,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,1,12,12,12
421,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,12,12,0
422,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,1,12,13,12


In [44]:
tn_vector_bigrams.shape

(424, 684)

In [45]:
tn_vector_bigrams = tn_vector_bigrams.drop_duplicates()
tn_vector_bigrams = tn_vector_bigrams.sample(n=len(fn_vector_bigrams), random_state=42)
tn_vector_bigrams.shape

(285, 684)

In [46]:
tn_base_final = tn_vector_bigrams.drop(['sucessor', 'predecessor','palavra'], axis=1)
tn_base_final

,AA,AB,AC,AD,AE,AF,AG,AH,AI,AJ,...,ZV,ZW,ZX,ZY,ZZ,y_predecessor,y_sucessor,cg_p,cg_w,cg_s
71,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,12,12,12
223,0,0,0,1,0,1,0,0,0,0,...,0,0,0,0,0,0,0,12,13,9
265,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,12,13,12
33,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,12,0,12
43,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,13,12,12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
369,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,13,13,0
210,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,12,13
261,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,12,12,12
350,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,12,12,0


In [47]:
fn_base_final.to_csv('base_fn_cg.csv', index=False)
tn_base_final.to_csv('base_tn_cg.csv', index=False)

# Passo 4 - Armazenamento das bases prontas para treino dos Falsos Negativos e Verdadeiros Negativos

no fim temos estes dois arquivos

duas tabelas contendo

Parte 1: Vetor de bigramas do token (O FALSO NEGATIVO OU VERDADEIRO NEGATIVO)  
Parte 2: A resultado da predição do nosso modelo para com os predecessores e sucessores do token representado no vetor de bigramas  
Parte 3: A classificação gramatical do predecessor, da token em si e do sucessor deste token.  

In [48]:
import pandas as pd

base = pd.read_csv(r"base_fn_cg.csv")
base

,AA,AB,AC,AD,AE,AF,AG,AH,AI,AJ,...,ZV,ZW,ZX,ZY,ZZ,y_predecessor,y_sucessor,cg_p,cg_w,cg_s
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,12,0,13
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,12,13,12
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,12,12,12
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,1,12,12,13
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,12,12,12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
280,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,12,12,0
281,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,1,12,12,12
282,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,12,12,0
283,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,1,12,13,12


In [49]:
import pandas as pd

base = pd.read_csv(r"base_tn_cg.csv")
base

,AA,AB,AC,AD,AE,AF,AG,AH,AI,AJ,...,ZV,ZW,ZX,ZY,ZZ,y_predecessor,y_sucessor,cg_p,cg_w,cg_s
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,12,12,12
1,0,0,0,1,0,1,0,0,0,0,...,0,0,0,0,0,0,0,12,13,9
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,12,13,12
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,12,0,12
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,13,12,12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
280,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,13,13,0
281,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,12,13
282,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,12,12,12
283,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,12,12,0
